# Retail Business Intelligence Through Data Mining
## Dataset 1: Superstore Sales
Steps: (1) load, (2) clean, (3) BI-style analysis, (4) classification of loss-making order lines, (5) charts.
**How to run on Kaggle:** create a new Notebook, click *Add Input*, search "Superstore" and add the dataset
containing `Sample - Superstore.csv`, then run all cells.

In [1]:
# 1. SETUP AND DATA LOADING
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)
warnings.filterwarnings("ignore")
try:
    from IPython.display import display
except ImportError:
    display = print

# Find the file automatically (Kaggle input folder first, then the current folder)
candidates = []
for root in ["/kaggle/input", "."]:
    candidates += glob.glob(os.path.join(root, "**", "*uperstore*.csv"), recursive=True)
if not candidates:
    raise FileNotFoundError("Superstore CSV not found. Add the dataset via 'Add Input' on Kaggle.")
SRC = candidates[0]
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "outputs"
os.makedirs(OUT, exist_ok=True)
print("Using file:", SRC)

raw = pd.read_csv(SRC, encoding="latin-1")
print("Shape:", raw.shape)
print("Missing cells:", int(raw.isna().sum().sum()))
display(raw.head())

FileNotFoundError: Superstore CSV not found. Add the dataset via 'Add Input' on Kaggle.

In [ ]:
# 2. DATA CLEANING AND PREPARATION
df = raw.copy()

# Remove duplicate order lines (identical in every column except the row number)
cols_wo_id = [c for c in df.columns if c != "Row ID"]
n_dup = int(df.duplicated(subset=cols_wo_id).sum())
df = df.drop_duplicates(subset=cols_wo_id).copy()

# Convert dates (the file uses month/day/year)
df["Order Date"] = pd.to_datetime(df["Order Date"], format="%m/%d/%Y")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], format="%m/%d/%Y")

# Consistency checks
bad_dates = int(df[["Order Date", "Ship Date"]].isna().sum().sum())
ship_before_order = int((df["Ship Date"] < df["Order Date"]).sum())
multi_name = int((df.groupby("Product ID")["Product Name"].nunique() > 1).sum())

# Postal codes as 5-character text, plus new analytical columns
df["Postal Code"] = df["Postal Code"].astype(str).str.zfill(5)
df["Year"] = df["Order Date"].dt.year
df["Month"] = df["Order Date"].dt.to_period("M").dt.to_timestamp()
df["Ship Days"] = (df["Ship Date"] - df["Order Date"]).dt.days
df["Loss"] = (df["Profit"] < 0).astype(int)          # target variable: 1 = loss-making line

print(f"Duplicate lines removed : {n_dup}")
print(f"Invalid dates           : {bad_dates}")
print(f"Shipped before ordered  : {ship_before_order}")
print(f"Product IDs with >1 name: {multi_name}")
print(f"Rows after cleaning     : {len(df)}")
df.to_csv(f"{OUT}/superstore_cleaned.csv", index=False)   # original file is left unchanged

In [ ]:
# 3. BUSINESS INTELLIGENCE ANALYSIS (KPIs)
print("Orders:", df["Order ID"].nunique(), "| Customers:", df["Customer ID"].nunique(),
      "| Products:", df["Product ID"].nunique())
print(f"Total sales : {df.Sales.sum():,.2f}")
print(f"Total profit: {df.Profit.sum():,.2f}  (margin {df.Profit.sum()/df.Sales.sum()*100:.2f}%)")
print(f"Loss-making lines: {df.Loss.sum()} ({df.Loss.mean()*100:.2f}%), total loss {df.loc[df.Loss==1,'Profit'].sum():,.2f}")
print(f"Correlation of discount with profit: {df.Discount.corr(df.Profit):.3f}")

def kpi(g):
    return pd.Series({"Sales": g.Sales.sum(), "Profit": g.Profit.sum(),
                      "Margin %": g.Profit.sum() / g.Sales.sum() * 100,
                      "Lines": len(g), "Loss %": g.Loss.mean() * 100})

by_year = df.groupby("Year").apply(kpi).round(2)
by_cat = df.groupby("Category").apply(kpi).round(2)
by_reg = df.groupby("Region").apply(kpi).round(2)
by_seg = df.groupby("Segment").apply(kpi).round(2)
by_sub = df.groupby("Sub-Category").apply(kpi).round(2).sort_values("Profit")
disc_bins = pd.cut(df.Discount, [-0.001, 0, 0.1, 0.2, 0.3, 0.5, 1.0],
                   labels=["0%", "1-10%", "11-20%", "21-30%", "31-50%", ">50%"])
by_disc = df.groupby(disc_bins).apply(kpi).round(2)

for title, t in [("By year", by_year), ("By category", by_cat), ("By region", by_reg),
                 ("By segment", by_seg), ("By sub-category (lowest to highest profit)", by_sub),
                 ("By discount level", by_disc)]:
    print(f"\n== {title} ==")
    display(t)

In [ ]:
# 4. DATA MINING: CLASSIFICATION OF LOSS-MAKING ORDER LINES
# Logistic regression and a decision tree predict whether an order line makes a loss.
num_f = ["Discount", "Quantity", "LogSales"]
cat_f = ["Category", "Sub-Category", "Region", "Segment", "Ship Mode"]
X = df[["Discount", "Quantity", "Sales"] + cat_f].copy()
X["LogSales"] = np.log1p(X.Sales)
X = X.drop(columns="Sales")
y = df["Loss"]

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
pre = ColumnTransformer([("num", StandardScaler(), num_f),
                         ("cat", OneHotEncoder(handle_unknown="ignore"), cat_f)])
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, min_samples_leaf=20, random_state=42),
}
skf = StratifiedKFold(5, shuffle=True, random_state=42)
rows, fitted, confs = {}, {}, {}
for name, m in models.items():
    pipe = Pipeline([("pre", pre), ("m", m)]).fit(Xtr, ytr)
    pred = pipe.predict(Xte)
    prob = pipe.predict_proba(Xte)[:, 1]
    cv = cross_val_score(Pipeline([("pre", pre), ("m", m)]), X, y, cv=skf, scoring="accuracy")
    rows[name] = {"Accuracy": accuracy_score(yte, pred), "Precision": precision_score(yte, pred),
                  "Recall": recall_score(yte, pred), "F1": f1_score(yte, pred),
                  "ROC AUC": roc_auc_score(yte, prob), "CV Accuracy (5-fold)": cv.mean()}
    confs[name] = confusion_matrix(yte, pred)
    fitted[name] = pipe

results = pd.DataFrame(rows).T.round(3)
print(f"Test records: {len(yte)} | Baseline accuracy (always 'profitable'): {max(yte.mean(), 1-yte.mean()):.3f}")
display(results)
for n, c in confs.items():
    print(f"\nConfusion matrix, {n} (rows = actual, columns = predicted)\n", c)

tree = fitted["Decision Tree"]
fn = [i.replace("num__", "").replace("cat__", "") for i in tree.named_steps["pre"].get_feature_names_out()]
importance = pd.Series(tree.named_steps["m"].feature_importances_, index=fn).sort_values(ascending=False)
print("\nDecision tree feature importance (top 8)")
display(importance.head(8).round(3))

In [ ]:
# 5. VISUALIZATIONS
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold"})
BLUE, RED, GREEN = "#2b6cb0", "#c53030", "#2f855a"

# Figure 1: monthly sales and profit
m = df.groupby("Month")[["Sales", "Profit"]].sum()
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(m.index, m.Sales / 1000, color=BLUE, lw=2, label="Sales")
ax.plot(m.index, m.Profit / 1000, color=GREEN, lw=2, label="Profit")
ax.set_title("Monthly Sales and Profit (2014-2017)"); ax.set_ylabel("Amount (thousands)")
ax.legend(frameon=False); ax.grid(axis="y", alpha=.25)
fig.tight_layout(); fig.savefig(f"{OUT}/fig1_monthly_sales_profit.png", dpi=150); plt.show()

# Figure 2: profit by sub-category
fig, ax = plt.subplots(figsize=(8, 5.2))
ax.barh(by_sub.index, by_sub.Profit / 1000, color=[RED if v < 0 else BLUE for v in by_sub.Profit])
ax.set_title("Total Profit by Sub-Category"); ax.set_xlabel("Profit (thousands)")
ax.axvline(0, color="black", lw=.8); ax.grid(axis="x", alpha=.25)
fig.tight_layout(); fig.savefig(f"{OUT}/fig2_profit_by_subcategory.png", dpi=150); plt.show()

# Figure 3: share of loss-making lines by discount level
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(by_disc.index.astype(str), by_disc["Loss %"], color=[BLUE if v < 50 else RED for v in by_disc["Loss %"]])
for i, v in enumerate(by_disc["Loss %"]):
    ax.text(i, v + 1.5, f"{v:.0f}%", ha="center")
ax.set_title("Share of Loss-Making Order Lines by Discount Level")
ax.set_xlabel("Discount"); ax.set_ylabel("Loss-making lines (%)"); ax.set_ylim(0, 110)
fig.tight_layout(); fig.savefig(f"{OUT}/fig3_loss_by_discount.png", dpi=150); plt.show()

# Figure 4: decision tree feature importance
top = importance.head(8)[::-1]
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(top.index, top.values, color=BLUE)
ax.set_title("Decision Tree: Most Important Features"); ax.set_xlabel("Importance")
fig.tight_layout(); fig.savefig(f"{OUT}/fig4_feature_importance.png", dpi=150); plt.show()
print("Files saved in:", OUT)